# Pokémon Battle Outcome Prediction

This notebook presents the three approaches submitted to the FDS Pokémon Battles Prediction 2026
competition. The task is to predict whether Player 1 wins a Generation 2 OU battle from its first
30 active turns. The dataset contains 10,000 labelled training battles and 5,000 test battles;
performance is measured by classification accuracy.

The notebook clones the accompanying GitHub repository and records the exact code commit used
for execution. Each approach runs in a separate Python process, and each section describes the
method and provides its execution cell.

### Approaches

| Approach | Main idea | Kaggle submission |
|---|---|---|
| Hidden knockouts and P1 history | Estimate hidden knockouts and encode Player 1's team history. | 57708941 |
| Turn-31 forecast, V5 | Forecast the next turn and blend CatBoost, XGBoost and LightGBM. | 57696039 |
| Turn-31 forecast, V2 | Forecast the next turn and stack boosted trees with logistic regression. | 57675780 |

## 1. Retrieve the implementation

The following cell clones the public repository and checks out `REPO_REF`. The resolved commit
is printed and recorded with each generated submission. A full commit SHA fixes the source
version for repeated executions.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = os.environ.get("FDS_REPO_URL", "https://github.com/teoang/fds-pokemon-challenge.git")
REPO_REF = os.environ.get("FDS_REPO_REF", "df8965555648fcd93cbaf5ac3425256f00199cfa")
WORKSPACE = Path(os.environ.get(
    "FDS_NOTEBOOK_WORK",
    "/kaggle/working" if Path("/kaggle/working").exists()
    else str(Path.cwd() / "outputs/local/notebook_workspace"),
)).resolve()
WORKSPACE.mkdir(parents=True, exist_ok=True)
REPO = WORKSPACE / "fds-pokemon-code"
git_env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
if not REPO.exists():
    subprocess.run(["git", "clone", "--quiet", "--no-checkout", REPO_URL, str(REPO)],
                   check=True, env=git_env)
else:
    if not (REPO / ".git").is_dir():
        raise RuntimeError(f"{REPO} exists and is not a Git clone")
    remote = subprocess.check_output(
        ["git", "-C", str(REPO), "remote", "get-url", "origin"], text=True).strip()
    dirty = subprocess.check_output(
        ["git", "-C", str(REPO), "status", "--porcelain"], text=True).strip()
    if remote != REPO_URL or dirty:
        raise RuntimeError("The existing checkout has another origin or local changes; use a new WORKSPACE")
subprocess.run(["git", "-C", str(REPO), "fetch", "--quiet", "origin", REPO_REF],
               check=True, env=git_env)
subprocess.run(["git", "-C", str(REPO), "checkout", "--quiet", "--detach", "FETCH_HEAD"],
               check=True, env=git_env)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
sys.path.insert(0, str(REPO))
print("Code commit:", COMMIT)

### Install dependencies

Package versions are pinned in `requirements-notebook.txt`. Each execution records its Python
and package versions in the execution manifest.

In [ ]:
if os.environ.get("FDS_SKIP_INSTALL") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r",
                    str(REPO / "requirements-notebook.txt")], check=True)
print("Environment setup complete")

## 2. Load the competition inputs

On Kaggle, attach `fds-pokemon-battles-prediction-2026` as the competition data source and enable
Internet access for the repository clone and dependency installation. The input directory must
contain `train.jsonl`, `test.jsonl`, `type_chart_gen2.csv` and `sample_submission.csv`.

The controls below select the input directory and approaches to execute. All three approaches
are selected by default; individual approaches can also be inspected separately. After changing
the selection, rerun the validation cell and the subsequent execution cells.

Generated files are written under `OUTPUT_ROOT`. The input files are read without modification.

In [ ]:
import json
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, Markdown, FileLink
from submission_tools.runner import SUBMISSIONS, resolve_data, preflight, bundle

try:
    initial_data = str(resolve_data())
except FileNotFoundError:
    initial_data = os.environ.get("FDS_DATA", "")
labels = [("1 — Hidden knockouts + P1 history", SUBMISSIONS[0]),
          ("2 — Turn-31 forecast, CatBoost-dominant V5", SUBMISSIONS[1]),
          ("3 — Turn-31 forecast, stacked V2", SUBMISSIONS[2])]
default_selection = tuple(filter(None, os.environ.get(
    "FDS_SUBMISSIONS", ",".join(SUBMISSIONS)).split(",")))
selection = widgets.SelectMultiple(options=labels, value=default_selection,
                                   description="Approaches", rows=3,
                                   layout=widgets.Layout(width="650px"))
data_input = widgets.Text(value=initial_data, description="Data folder",
                          layout=widgets.Layout(width="850px"))
OUTPUT_ROOT = Path(os.environ.get("FDS_OUTPUT_ROOT", str(WORKSPACE / "final_submissions"))).resolve()
display(selection, data_input)
print("Outputs:", OUTPUT_ROOT)

## 3. Validate inputs and source files

Before training, the notebook checks the required input files and verifies the archived source
and implementation checksums. The table below summarizes these checks for the selected approaches.

In [ ]:
selected = tuple(selection.value)
if not selected:
    raise ValueError("Select at least one approach")
DATA = resolve_data(data_input.value.strip() or None)
checks = [preflight(REPO / "submissions" / name, DATA) for name in selected]
results = {}
display(pd.DataFrame([{
    "Approach": check["submission"], "Ready to execute": check["ready_to_execute"],
    "Source issues": "; ".join(check["errors"]),
} for check in checks]))
if any(not check["ready_to_execute"] for check in checks):
    raise RuntimeError("Execution stopped: a selected approach has incomplete source code. See the validation table.")


def run_selected(name):
    if name not in selected:
        print("Not selected:", name)
        return
    results.pop(name, None)
    out = OUTPUT_ROOT / name
    env = dict(os.environ, FDS_DATA=str(DATA), MPLCONFIGDIR=str(WORKSPACE / "matplotlib_cache"))
    subprocess.run([sys.executable, "-u", str(REPO / "submissions" / name / "run.py"),
                    "--data", str(DATA), "--out", str(out)],
                   check=True, cwd=REPO, env=env)
    record = json.loads((out / "run_manifest.json").read_text())
    if record["git_commit"] != COMMIT:
        raise RuntimeError("The model ran from another code commit")
    results[name] = record
    display(FileLink(os.path.relpath(out / "submission.csv", Path.cwd())))

## 4. Submission 1: hidden knockouts and Player 1 history

Player 2's knockouts are often absent from the recorded fainting states. We estimate these
hidden knockouts using Player 1's switches as a control group, then recompute the expected
state at turn 30. A LightGBM classifier combines that state with Player 1's species and items,
move counts, and an out-of-fold encoding of the exact Player 1 team.

Validation uses stratified five-fold cross-validation repeated three times. The final classifier
is fitted on all training battles. This cell runs the final configuration with its original
features, seeds and model settings; the separate ablation experiments are described in the report.

**Reference Kaggle submission:** 57708941.

In [ ]:
run_selected(SUBMISSIONS[0])

## 5. Submission 2: turn-31 forecast, V5

This approach first predicts each player's next switch and the types of the next active Pokémon.
Those predictions are combined with battle-state features, observed matchups, move information
and target-encoded win-rate features.

LightGBM, XGBoost and CatBoost are trained using ten-fold stratified cross-validation. Their test
probabilities are combined with weights of 0.08, 0.12 and 0.80, respectively. A probability
threshold of 0.5 gives the binary predictions. The original feature definitions, model settings,
blend weights and threshold are preserved.

**Reference Kaggle submission:** 57696039.

In [ ]:
run_selected(SUBMISSIONS[1])

## 6. Submission 3: turn-31 forecast, V2

The submitted V2 approach combines turn-31 forecasts with features describing the observed
battle. It trains LightGBM, XGBoost and CatBoost using ten-fold stratified cross-validation,
then combines their out-of-fold probabilities with a logistic-regression meta-learner.

The classification threshold averages 0.5 with a threshold chosen to match the training win-rate
prior. The turn-level module builds the forecasting dataset and fits the switch and next-type
classifiers.

**Reference Kaggle submission:** 57675780.

In [ ]:
run_selected(SUBMISSIONS[2])

## 7. Verify and export predictions

Each generated CSV contains one row per test battle, with columns `battle_id` and `player_won`.
Predictions are encoded as 1 for a Player 1 win and 0 for a Player 1 loss. Validation checks
that the battle identifiers match `sample_submission.csv`, are unique and have binary predictions.

The generated predictions are compared with an order-independent SHA-256 fingerprint of the
corresponding CSV submitted to Kaggle. The reference predictions are used only for verification.

A combined archive is exported when all three approaches have completed using the same code
commit and inputs, and all three prediction sets match their references. The archive contains
the three regenerated CSVs and a combined execution manifest.

In [ ]:
if results:
    display(pd.DataFrame([{
        "Approach": name, "Rows": record["output"]["rows"],
        "Original submission": record["reference"]["submission_ref"],
    } for name, record in results.items()]))
if set(results) != set(SUBMISSIONS):
    display(Markdown("Run all three approaches to create the combined archive."))
else:
    archive = bundle(list(results.values()), OUTPUT_ROOT)
    display(Markdown("The combined archive contains the three CSVs and their execution manifest."))
    display(FileLink(os.path.relpath(archive, Path.cwd())))

## Code and execution records

The [GitHub repository](https://github.com/teoang/fds-pokemon-challenge) contains the implementation
and original source archives for all three approaches. Each folder under `submissions/` includes
`source.json`, which records source checksums and execution order, and `reference.json`, which
identifies the corresponding Kaggle submission and its prediction fingerprint.

For each completed execution, `run_manifest.json` records the repository commit, input-file
checksums, runtime versions and verification result. The execution log and additional model
outputs are saved alongside the generated CSV.